# Torsional buckling of ring- and stringer-stiffened cylinders: Moore and Wescoat (1944), Table II

Moore and Wescoat [1] tested in torsion 24S-T cylinders of mean diameter
$D = 30.08$ in with $1/2 \times 1/2$ in solid rings inside the skin and 8
or 16 channel stringers outside (specimens 19, 14, 20, 15 and 21), and
gave in Table II (p. 26) the mean observed shear stress at which the
panels between rings and stringers buckled, panel by panel, with the
values computed from four theories: ANC-5 [2], Kromm [3], Ebner [4] and
Donnell [5] (the last for an unstiffened cylinder of the length of the
bay). This notebook computes the first buckling stress of the perfect
stiffened cylinders with the BFSC cylinder element of Sanders-type
kinematics for the skin and BFSC plate elements for the stringers, and
compares it with the lowest panel value of Table II and with the four
theories. The ring-only specimens of the same report (Table I) are the
subject of the notebook `moore1944_ring_cylinders.ipynb`, whose model of
the rings, boundary conditions and pre-buckling state are used here.

Pages are the printed ones of [1] (pdf page = printed page + 2).

**References**

1. Moore, R. L., and Wescoat, C., Torsion tests of stiffened circular
   cylinders, NACA ARR 4E31 (Wartime Report W-89), 1944.
2. Strength of Aircraft Elements, Army-Navy-Civil Committee on Aircraft
   Design Criteria (ANC-5), 1942 (reference 1 of [1]).
3. Kromm, A., The limit of stability of a curved plate strip under shear
   and axial stresses, NACA TM 898, 1939 (reference 3 of [1]).
4. Ebner, H., The strength of shell bodies - theory and practice, NACA TM
   838, 1937 (reference 4 of [1]).
5. Donnell, L. H., Stability of thin-walled tubes under torsion, NACA
   Report 479, 1933 (reference 6 of [1]).

In [1]:
import sys
import time
from multiprocessing import Pool
sys.path.append('.')

import numpy as np
import matplotlib.pyplot as plt

import moore1944_utils as mu

## Data

From [1], in addition to the data of the ring-only notebook ($E = 10.5
\times 10^6$ psi, $\nu = 1/3$, p. 7; $R = 15.04$ in; length between the
bulkhead centres $L = 35.625$ in, Fig. 11; rings inside the skin):

- sheet thicknesses and number of stringers, Table II: 19, 14 and 20,
  $t = 0.020$ in; 15, 0.0195 in; 21, 0.0205 in; 8 stringers 11.81 in
  apart (19, 14) or 16 stringers 5.90 in apart (20, 15, 21), one of them
  at the longitudinal seam;
- ring stations, Fig. 11 (bays between bulkhead and ring centres, below);
- stringers outside the skin, 0.032 in 24S-T channels (Fig. 30, p. 56):
  overall width 1.12 in, overall height 0.64 in, bend radius 1/8 in, a
  curled lip at the free edge of each web, $A = 0.072$ in$^2$, $I_{XX} =
  0.00375$ in$^4$, $I_{YY} = 0.0063$ in$^4$; base 1/2 in wide, riveted to
  the skin by one row of 1/8 in rivets at its centre (Fig. 1).

Table II, the lowest panel of each specimen (stresses in psi):

In [2]:
print('| spec | t (in) | stringers | bays (in) | lowest panel (in) | observed | ANC-5 | Kromm | Ebner | Donnell |')
print('|---|---|---|---|---|---|---|---|---|---|')
for s, row in mu.TABLE_II.items():
    sp = mu.SPECIMENS[s]
    f = lambda v: '-' if v is None else '%d' % v
    print('| %s | %.4g | %d | %s | %.2f x %.2f | %s |' % (
        s, sp['t'], sp['ns'], ' / '.join('%.4g' % b for b in sp['bays']), row[0],
        2*np.pi*mu.R/sp['ns'], ' | '.join(f(v) for v in row[1:])))

| spec | t (in) | stringers | bays (in) | lowest panel (in) | observed | ANC-5 | Kromm | Ebner | Donnell |
|---|---|---|---|---|---|---|---|---|---|
| 19 | 0.02 | 8 | 4.312 / 13.5 / 13.5 / 4.312 | 13.50 x 11.81 | 2110 | 1620 | 1810 | 2320 | 2210 |
| 14 | 0.02 | 8 | 4.312 / 4.5 / 9 / 9 / 4.5 / 4.312 | 9.00 x 11.81 | 2950 | 1750 | 2700 | 2960 | 2770 |
| 20 | 0.02 | 16 | 4.312 / 13.5 / 13.5 / 4.312 | 13.50 x 5.91 | 2870 | 2070 | 2600 | 2570 | 2210 |
| 15 | 0.0195 | 16 | 8.812 / 9 / 9 / 8.812 | 8.81 x 5.91 | 3370 | 2110 | - | - | 2680 |
| 21 | 0.0205 | 16 | 2.062 / 6.75 / 9 / 9 / 6.75 / 2.062 | 9.00 x 5.91 | 4120 | 2240 | - | - | 2910 |


For specimen 15 the values of Kromm and Ebner, 2880 and 3170 psi, are
printed between the rows of the 8.81 and the 9.00 in panels; the ratios of
Table III (p. 25), 3590/2880 = 1.24 and 3590/3170 = 1.13, place them in
the row of the 9.00 in panels, and Table III gives no ratio for the 8.81 in
row. Donnell's values are reproduced by the formula of the ring-only
notebook with $L$ the panel length and the sheet thickness of the
specimen: 2204, 2775, 2204, 2720 and 2865 psi against the printed 2210,
2770, 2210, 2680 and 2910 (within 1.6 %).

In [3]:
for s, row in mu.TABLE_II.items():
    t = mu.SPECIMENS[s]['t']
    print('%-3s Donnell L = %.2f in: %.0f psi, Table II %d psi' % (s, row[0], mu.donnell_tau(t, row[0]), row[5]))

19  Donnell L = 13.50 in: 2204 psi, Table II 2210 psi
14  Donnell L = 9.00 in: 2775 psi, Table II 2770 psi
20  Donnell L = 13.50 in: 2204 psi, Table II 2210 psi
15  Donnell L = 8.81 in: 2720 psi, Table II 2680 psi
21  Donnell L = 9.00 in: 2865 psi, Table II 2910 psi


## The stringer section

The model of the channel (`channel_cylinder_sector.ChannelCylinderSector`)
has a base on the skin, two webs normal to the skin and a flat lip at the
free edge of each web, bent at 90 degrees away from the channel, all of
thickness $t_c = 0.032$ in:

- the base is the outer layer of a two-layer laminate skin + base between
  the webs, $b = 0.5 - 0.032 = 0.468$ in between the middle surfaces of the
  webs (outer width 1/2 in, Fig. 1);
- the webs go from the middle surface of the skin to the middle surface of
  the lips, $H_w = t/2 + 0.64 - 0.032/2$ in (overall height 0.64 in from
  the outer surface of the skin);
- the curled lips of Fig. 30 are replaced by flat lips. Their width $b_l$,
  from the middle surface of the web to the free edge, is chosen so that
  the area of the section (middle lines, square corners) is the $A =
  0.072$ in$^2$ of Fig. 30: $b_l = (A/t_c - b - 2(0.64 - t_c))/2 = 0.283$
  in. The overall width is then $b + 2 b_l + t_c = 1.07$ in against 1.12
  in.

In [4]:
for bl, label in [(mu.lip_width_for_area(), 'area of Fig. 30'),
                  ((mu.CH_WIDTH - mu.CH_T)/2 - (mu.CH_BASE - mu.CH_T)/2, 'overall width of Fig. 30')]:
    sec = mu.channel_section(bl)
    print('b_l = %.3f in (%s): A = %.4f in2 (%+.1f %%), centroid %.3f in above the skin, '
          'I_XX = %.5f in4 (%+.1f %%), I_YY = %.5f in4 (%+.1f %%)' % (
          bl, label, sec['A'], 100*(sec['A']/mu.CH_A - 1), sec['zc'],
          sec['Ixx'], 100*(sec['Ixx']/mu.CH_IXX - 1), sec['Iyy'], 100*(sec['Iyy']/mu.CH_IYY - 1)))

b_l = 0.283 in (area of Fig. 30): A = 0.0720 in2 (+0.0 %), centroid 0.333 in above the skin, I_XX = 0.00425 in4 (+13.2 %), I_YY = 0.00508 in4 (-19.3 %)
b_l = 0.310 in (overall width of Fig. 30): A = 0.0737 in2 (+2.4 %), centroid 0.340 in above the skin, I_XX = 0.00439 in4 (+17.1 %), I_YY = 0.00557 in4 (-11.6 %)


With the area of Fig. 30, $I_{XX}$ is 13 % above and $I_{YY}$ 19 % below
the values of Fig. 30: the curls of the real section bring material down
towards the base (lower $I_{XX}$) and out to the 1.12 in overall width
(higher $I_{YY}$). The differences are second order for the buckling of the
skin panels, which the stringers support mainly through their bending
stiffness normal to the skin; the lip width of the overall width, $b_l =
0.31$ in, is a sensitivity below.

## Model

`moore1944_utils.MooreSector` with the channel: a sector of one stringer
pitch, $\phi = 2\pi/n_s$, the stringer at its middle, the edge $\theta =
\phi$ tied to $\theta = 0$ by $\boldsymbol{u}(\phi) = e^{i j \phi}
\boldsymbol{u}(0)$, one Hermitian problem per Bloch index $j = 0, 1,
\ldots, n_s/2$ (indices $j$ and $n_s - j$ have the same multipliers). The
index $j$ contains the circumferential harmonics $n = |j + k n_s|$ coupled
by the stringers. The torque gives the constant membrane shear $N_{xy} =
T/(2\pi R^2)$ in the skin, the stringers and the rings unloaded (the exact
membrane state, checked with a prescribed end rotation in the ring-only
notebook); the geometric stiffness matrix is that of the skin elements
with a constant $N_{xy}$, so the plates of the stringers have none. The
lowest positive multiplier over all indices is the first buckling stress
$\tau_{cr} = N_{xy}/t$.

- Rings as in the ring-only notebook: a band of skin elements of axial
  width 1/2 in whose laminate is the skin, the ring layer of 1/2 in on the
  inner face ($\nu = 0$) and, under the stringer, the base on the outer face.
- Stringers: continuous bond of the base to the skin (the two-layer
  laminate) and rigid 90-degree corners web-base and web-lip; the single
  row of rivets is not modelled.
- Ends: hinged (baseline), $u = v = w = 0$ and their derivatives along the
  edge for the skin, and the displacements $U = V = W = 0$ and their
  derivatives along the edge for the webs and lips (the stringers attached
  to the bulkheads); clamped adds $w_{,x} = w_{,xy} = 0$ ($W_{,X} =
  W_{,XY} = 0$ of the plates). Rigid rings: $v = w = 0$ at the line of the
  ring centres.

## Mesh convergence

A mesh is `(dx, da, nweb, nlip)`: largest element length along $x$
between the ring bands, largest element arc of the skin between the
stringers (`nbay` elements from the sector edge to a web), elements along
a web and across a lip; two elements across the base and across a ring
band. All Bloch indices of each specimen are computed, in parallel.

In [5]:
specs = ['19', '14', '20', '15', '21']
NPROC = 8


def nbay(spec, da):
    ns = mu.SPECIMENS[spec]['ns']
    half = np.pi*mu.R/ns - (mu.CH_BASE - mu.CH_T)/2
    return int(np.ceil(half/da))


def tasks(spec, mesh, **kw):
    dx, da, nweb, nlip = mesh
    ns = mu.SPECIMENS[spec]['ns']
    out = []
    for j in range(ns//2 + 1):
        d = dict(spec=spec, dx=dx, nbay=nbay(spec, da), js=[j])
        if kw.get('model') != 'band':
            d.update(nweb=nweb, nlip=nlip)
        d.update(kw)
        out.append(d)
    return out


def run(groups):
    # groups: dict key -> list of task dictionaries; returns key -> result
    flat = [(key, t) for key, ts in groups.items() for t in ts]
    with Pool(NPROC) as pool:
        res = pool.map(mu.lowest, [t for _, t in flat])
    out = {}
    for (key, _), (kw, r) in zip(flat, res):
        o = out.setdefault(key, dict(tau={}, time=0., ndof=0))
        j = list(r['tau'])[0]
        o['tau'][j] = r['tau'][j]
        o['time'] += r['time']
        o['ndof'] = max(o['ndof'], r['ndof'])
        if 'tau_min' not in o or r['tau_min'] < o['tau_min']:
            o.update(tau_min=r['tau_min'], j=j, x=r['x'], theta=r['theta'], w=r['w'])
    return out


meshes = [(0.7, 0.9, 3, 1), (0.5, 0.65, 4, 1), (0.35, 0.45, 5, 2)]
t0 = time.time()
conv = run({(s, m): tasks(s, m) for s in specs for m in meshes})
print('%.0f s' % (time.time() - t0))
print('%-5s' % 'spec' + ''.join('%32s' % str(m) for m in meshes))
for s in specs:
    row = '%-5s' % s
    prev = None
    for m in meshes:
        r = conv[s, m]
        d = '' if prev is None else '%+.2f%%' % (100*(r['tau_min']/prev - 1))
        row += '%32s' % ('%.1f (j=%d, %d dofs) %s' % (r['tau_min'], r['j'], r['ndof'], d))
        prev = r['tau_min']
    print(row)

628 s
spec                 (0.7, 0.9, 3, 1)               (0.5, 0.65, 4, 1)              (0.35, 0.45, 5, 2)
19          2639.6 (j=1, 13832 dofs)  2625.8 (j=1, 23340 dofs) -0.52% 2619.0 (j=1, 45276 dofs) -0.26%
14          3162.0 (j=4, 14824 dofs)  3143.3 (j=4, 23956 dofs) -0.59% 3133.6 (j=4, 46132 dofs) -0.31%
20          4000.2 (j=2, 10484 dofs)  3980.5 (j=2, 17276 dofs) -0.49% 3970.0 (j=2, 32580 dofs) -0.26%
15          4153.0 (j=3, 10860 dofs)  4129.9 (j=3, 17276 dofs) -0.56% 4117.7 (j=3, 32580 dofs) -0.30%
21          4458.1 (j=3, 11236 dofs)  4434.8 (j=3, 17732 dofs) -0.52% 4421.8 (j=3, 33196 dofs) -0.29%


Each entry is the lowest $\tau_{cr}$ (psi), its Bloch index, the number of
reduced degrees of freedom of one index and the change from the previous
mesh. The last refinement changes $\tau_{cr}$ by less than 1 % (table), and
the finest mesh gives the values below. The sensitivities are computed on
the second mesh and given as changes relative to the second mesh.

In [6]:
fine, mid = meshes[-1], meshes[1]
base = {s: conv[s, fine] for s in specs}
for s in specs:
    print('%-3s ' % s + '  '.join('j=%d: %.0f' % (j, v) for j, v in sorted(base[s]['tau'].items())))

19  j=0: 2620  j=1: 2619  j=2: 2619  j=3: 2621  j=4: 2623
14  j=0: 3145  j=1: 3143  j=2: 3139  j=3: 3135  j=4: 3134
20  j=0: 3997  j=1: 3983  j=2: 3970  j=3: 3971  j=4: 3986  j=5: 4014  j=6: 4051  j=7: 4090  j=8: 4123
15  j=0: 4149  j=1: 4138  j=2: 4125  j=3: 4118  j=4: 4119  j=5: 4127  j=6: 4141  j=7: 4159  j=8: 4170
21  j=0: 4452  j=1: 4442  j=2: 4428  j=3: 4422  j=4: 4426  j=5: 4438  j=6: 4456  j=7: 4479  j=8: 4493


## Sensitivities

On the second mesh: clamped ends; rigid ring supports; lips of the overall
width of Fig. 30 ($b_l = 0.31$ in); no stringers (rings only, the skin of
the specimen); each stringer represented by a band of the skin of the width
of the base whose laminate adds a layer with the axial stiffness, the
eccentricity and the bending stiffness $E I_{XX}$ of the idealised channel
(`moore1944_utils.SkinBandSector`, no plate elements, the model of the
test `tests/test_moore1944_torsion.py`).

In [7]:
bl31 = (mu.CH_WIDTH - mu.CH_T)/2 - (mu.CH_BASE - mu.CH_T)/2
sens = {
    'clamped': dict(ends='clamped'),
    'rigid rings': dict(rigid_rings=True),
    'lips 0.31 in': dict(bl=bl31),
    'no stringers': dict(stringers=False),
    'skin-band stringers': dict(model='band'),
}
t0 = time.time()
sres = run({(name, s): tasks(s, mid, **kw) for name, kw in sens.items() for s in specs})
print('%.0f s' % (time.time() - t0))
print('%-22s' % 'case' + ''.join('%24s' % s for s in specs))
print('%-22s' % 'hinged, second mesh' + ''.join('%24s' % ('%.0f (j=%d)' % (conv[s, mid]['tau_min'], conv[s, mid]['j'])) for s in specs))
for name in sens:
    row = '%-22s' % name
    for s in specs:
        r = sres[name, s]
        row += '%24s' % ('%.0f (j=%d) %+.1f%%' % (r['tau_min'], r['j'], 100*(r['tau_min']/conv[s, mid]['tau_min'] - 1)))
    print(row)

348 s
case                                        19                      14                      20                      15                      21
hinged, second mesh                 2626 (j=1)              3143 (j=4)              3981 (j=2)              4130 (j=3)              4435 (j=3)
clamped                       2626 (j=1) +0.0%        3143 (j=4) +0.0%        3981 (j=2) +0.0%        4132 (j=3) +0.0%        4435 (j=3) +0.0%
rigid rings                   2628 (j=1) +0.1%        3147 (j=4) +0.1%        3983 (j=2) +0.1%        4133 (j=4) +0.1%        4439 (j=3) +0.1%
lips 0.31 in                  2626 (j=1) +0.0%        3143 (j=4) +0.0%        3981 (j=2) +0.0%        4130 (j=3) +0.0%        4435 (j=3) +0.0%
no stringers                 2259 (j=4) -14.0%        2839 (j=1) -9.7%       2258 (j=4) -43.3%       2729 (j=7) -33.9%       2927 (j=7) -34.0%
skin-band stringers           2603 (j=2) -0.9%        3128 (j=4) -0.5%        3785 (j=3) -4.9%        3985 (j=4) -3.5%        4294 (j=4)

## Comparison with Table II

$\tau_{FE}$: finest mesh, hinged ends; the clamped value is the hinged one
times the clamped/hinged ratio of the second mesh. Differences relative to
the Table II values of the lowest panel.

In [8]:
def pct(a, b):
    return '-' if b is None else '%+.1f %%' % (100*(a/b - 1))


print('| spec | tau FE hinged | tau FE clamped | observed | ANC-5 | Kromm | Ebner | Donnell '
      '| FE/obs | FE/ANC-5 | FE/Kromm | FE/Ebner | FE/Donnell | clamped/obs |')
print('|---|---|---|---|---|---|---|---|---|---|---|---|---|---|')
final = {}
for s in specs:
    Lp, obs, anc, kromm, ebner, donn = mu.TABLE_II[s]
    h = base[s]['tau_min']
    c = h*sres['clamped', s]['tau_min']/conv[s, mid]['tau_min']
    final[s] = (h, c)
    f = lambda v: '-' if v is None else '%d' % v
    print('| %s | %.0f | %.0f | %d | %s | %s | %s | %s | %s | %s | %s | %s | %s | %s |' % (
        s, h, c, obs, f(anc), f(kromm), f(ebner), f(donn), pct(h, obs), pct(h, anc),
        pct(h, kromm), pct(h, ebner), pct(h, donn), pct(c, obs)))

| spec | tau FE hinged | tau FE clamped | observed | ANC-5 | Kromm | Ebner | Donnell | FE/obs | FE/ANC-5 | FE/Kromm | FE/Ebner | FE/Donnell | clamped/obs |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 19 | 2619 | 2619 | 2110 | 1620 | 1810 | 2320 | 2210 | +24.1 % | +61.7 % | +44.7 % | +12.9 % | +18.5 % | +24.1 % |
| 14 | 3134 | 3134 | 2950 | 1750 | 2700 | 2960 | 2770 | +6.2 % | +79.1 % | +16.1 % | +5.9 % | +13.1 % | +6.2 % |
| 20 | 3970 | 3970 | 2870 | 2070 | 2600 | 2570 | 2210 | +38.3 % | +91.8 % | +52.7 % | +54.5 % | +79.6 % | +38.3 % |
| 15 | 4118 | 4120 | 3370 | 2110 | - | - | 2680 | +22.2 % | +95.2 % | - | - | +53.6 % | +22.2 % |
| 21 | 4422 | 4422 | 4120 | 2240 | - | - | 2910 | +7.3 % | +97.4 % | - | - | +52.0 % | +7.3 % |


Torques, $T = \tau \pi D^2 t/24$ ft lb:

In [9]:
for s in specs:
    t = mu.SPECIMENS[s]['t']
    h, c = final[s]
    print('%-3s FE hinged %.0f ft lb, FE clamped %.0f ft lb, observed (lowest panel) %.0f ft lb' % (
        s, mu.torque_ftlb(h, t), mu.torque_ftlb(c, t), mu.torque_ftlb(mu.TABLE_II[s][1], t)))

19  FE hinged 6204 ft lb, FE clamped 6204 ft lb, observed (lowest panel) 4998 ft lb
14  FE hinged 7423 ft lb, FE clamped 7423 ft lb, observed (lowest panel) 6988 ft lb
20  FE hinged 9404 ft lb, FE clamped 9404 ft lb, observed (lowest panel) 6798 ft lb
15  FE hinged 9510 ft lb, FE clamped 9514 ft lb, observed (lowest panel) 7783 ft lb
21  FE hinged 10736 ft lb, FE clamped 10736 ft lb, observed (lowest panel) 10003 ft lb


## Buckling modes

Normal displacement $w$ of the skin (real part, arbitrary scale) of the
lowest mode around the whole circumference, finest mesh, hinged ends;
dashed: ring stations, dotted: stringers.

In [10]:
fig, axes = plt.subplots(len(specs), 1, figsize=(8, 2.4*len(specs)), sharex=True)
for ax, s in zip(axes, specs):
    r = base[s]
    w = r['w']/np.abs(r['w']).max()
    ax.contourf(r['theta']*mu.R, r['x'], w, levels=21, cmap='RdBu_r')
    for xr in mu.ring_stations(mu.SPECIMENS[s]['bays']):
        ax.axhline(xr, color='k', ls='--', lw=0.8)
    ns = mu.SPECIMENS[s]['ns']
    for k in range(ns):
        ax.axvline(k*2*np.pi*mu.R/ns, color='k', ls=':', lw=0.8)
    ax.set_ylabel('$x$ (in)')
    ax.set_title('specimen %s: j = %d, tau = %.0f psi' % (s, r['j'], r['tau_min']), fontsize=9)
axes[-1].set_xlabel(r'$R\theta$ from a stringer (in)')
fig.tight_layout()
plt.show()

## Discussion

- **Convergence.** The last refinement changes $\tau_{cr}$ by 0.26 to
  0.31 %; the sensitivities, on the second mesh, are changes of less than
  0.1 % (ends, rigid rings, lip width), far below the mesh error.
- **Stringers.** Without stringers the same skin and rings buckle 10 to 14 %
  lower with 8 stringers and 33 to 43 % lower with 16. The multipliers of
  the different Bloch indices are within 0.4 % (8 stringers) and 4 % (16
  stringers) of each other: the stringers act nearly as nodal lines of
  panel modes, and the lowest mode (figure) is a buckling of the skin
  panels between rings and stringers. The skin-band representation of the
  stringers (axial and bending stiffness normal to the skin only, no web
  or lip plates) is 0.5 to 0.9 % below the discrete channel with 8
  stringers and 3 to 5 % below with 16; the band has neither the lateral
  bending nor the torsional stiffness of the channel.
- **Ends and rings.** Clamped ends and rigid ring supports change
  $\tau_{cr}$ by 0.1 % at most: the rings bound the panels, as in the
  ring-only specimens.
- **Lips.** Flat lips of the area of Fig. 30 (0.283 in) or of its overall
  width (0.31 in) give the same $\tau_{cr}$ to 0.1 %: the section
  idealisation of the curls does not matter for the first buckling.
- **Tests.** The observed stresses of the lowest panel are 6 to 38 % below
  the finite element values of the perfect shell (14 and 21: 6 and 7 %;
  19, 15 and 20: 24, 22 and 38 %). The tests include initial
  imperfections, the riveted attachment of the stringers through a single
  row of rivets (continuous bond here) and the reading of the onset of
  buckling.
- **Moore's theories.** For the 8-stringer specimens Ebner's values are
  the closest to the finite element values (finite element 13 and 6 %
  above them for 19 and 14), followed by Donnell's unstiffened cylinder of
  the bay length (19 and 13 % above). For the 16-stringer specimens all the
  theories of Table II are 34 to 49 % below the finite element values; in
  the model the base of the channel, bonded over 1/2 in, narrows the panel
  and restrains its edges, and Donnell's value ignores the stringers
  (Table II, footnote).